# Stage 3D — MVTec leather validation and locked test evaluation (Kaggle)

This Kaggle version loads the completed 2,000-step checkpoint and performs the first reduced evaluation.

Before running, attach two notebook inputs:
1. The official MVTec AD data containing `leather/train/good`, `leather/test`, and `leather/ground_truth`.
2. A private dataset containing the Google Drive checkpoint file `latest.pt`.

Protocol decisions are fixed before reading test masks:
- `t_distance = 250`, following the fixed-time author evaluation routine and the paper's partial-diffusion analysis.
- The binary pixel threshold is the 99.5th percentile of reconstruction residuals across 49 held-out normal validation images.
- The official test set is evaluated once after the threshold is frozen.
- Training and validation use no anomalous images or masks.

This is a one-category, 2,000-step reduced experiment rather than a reproduction of the paper's 3,000-epoch result. Keep Kaggle Internet enabled so the notebook can obtain the five pinned, hash-verified author source files.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, random, shutil, subprocess, sys, time, zipfile
from collections import Counter, defaultdict

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT = Path('/kaggle/input')
checkpoint_candidates = sorted(KAGGLE_INPUT.rglob('latest.pt'))
if len(checkpoint_candidates) != 1:
    raise FileNotFoundError(
        'Expected exactly one latest.pt below /kaggle/input, found: '
        + repr([str(path) for path in checkpoint_candidates])
    )
CHECKPOINT = checkpoint_candidates[0]

def find_leather_categories(search_root):
    found = []
    for good_dir in search_root.rglob('good'):
        if not good_dir.is_dir() or good_dir.parent.name != 'train':
            continue
        candidate = good_dir.parent.parent
        if candidate.name == 'leather' and (candidate/'test').is_dir() and (candidate/'ground_truth').is_dir():
            found.append(candidate)
    return sorted(set(found))

category_candidates = find_leather_categories(KAGGLE_INPUT)
if not category_candidates:
    matching_archives = []
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        with zipfile.ZipFile(archive_path) as archive:
            normalized = ['/' + item.filename.replace('\\','/').lstrip('/') for item in archive.infolist()]
            if any('/leather/train/good/' in name for name in normalized):
                matching_archives.append(archive_path)
    if len(matching_archives) == 1:
        extraction_root = Path('/kaggle/working/uploaded_leather')
        extraction_root.mkdir(parents=True, exist_ok=True)
        resolved_root = extraction_root.resolve()
        with zipfile.ZipFile(matching_archives[0]) as archive:
            for item in archive.infolist():
                target = (extraction_root/item.filename).resolve()
                if target != resolved_root and resolved_root not in target.parents:
                    raise ValueError(f'Unsafe ZIP member: {item.filename}')
            archive.extractall(extraction_root)
        print('Extracted leather archive:', matching_archives[0])
        category_candidates = find_leather_categories(extraction_root)
    elif len(matching_archives) > 1:
        raise FileNotFoundError('More than one uploaded ZIP contains leather data: ' + repr([str(p) for p in matching_archives]))
if len(category_candidates) != 1:
    raise FileNotFoundError(
        'Expected exactly one leather category below /kaggle/input, found: '
        + repr([str(path) for path in category_candidates])
    )
CATEGORY_ROOT = category_candidates[0]
DATA_ROOT = CATEGORY_ROOT.parent

WORK = Path('/kaggle/working/labelinspect')
OUTPUT = WORK/'mvtec_leather_evaluation'
OUTPUT.mkdir(parents=True, exist_ok=True)

SEED = 230224
IMAGE_SIZE = 224
BATCH_SIZE = 4
T_DISTANCE = 250
TARGET_NORMAL_PIXEL_FPR = 0.005
print('GPU:', torch.cuda.get_device_name(0))
print('Leather category:', CATEGORY_ROOT)
print('Checkpoint:', CHECKPOINT)
print('Output:', OUTPUT)


In [ ]:
missing=[]
for package,module in [('timm','timm'),('einops','einops'),('numba','numba'),('scikit-learn','sklearn')]:
    if importlib.util.find_spec(module) is None: missing.append(package)
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Dependencies ready.')


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
spec=importlib.util.spec_from_file_location('labelinspect_author_smoke',WORK/'author_smoke.py')
author=importlib.util.module_from_spec(spec);sys.modules[spec.name]=author;spec.loader.exec_module(author)
import numba
numba.set_num_threads(min(2,numba.get_num_threads()))
source_cache=WORK/'upstream'/author.COMMIT
hashes=author.fetch_sources(source_cache)
model_module,diffusion_ns,Adapter,compatibility=author.load_author_components(source_cache,OUTPUT)
print('Pinned author commit:',author.COMMIT)


In [ ]:
def locate_category(root:Path,category='leather'):
    candidates=[root/category,root/'mvtec_anomaly_detection'/category]
    if root.name==category:candidates.insert(0,root)
    for candidate in candidates:
        if (candidate/'train'/'good').is_dir() and (candidate/'test').is_dir():return candidate
    raise FileNotFoundError(f'Could not find leather category below {root}.')

category_root=CATEGORY_ROOT
all_normal=sorted((category_root/'train'/'good').glob('*.png'))
assert len(all_normal)==245,f'Expected 245 train/good images, found {len(all_normal)}'
split_rng=random.Random(SEED);shuffled=all_normal.copy();split_rng.shuffle(shuffled)
val_count=max(1,round(len(shuffled)*.20));val_paths=sorted(shuffled[:val_count]);train_paths=sorted(shuffled[val_count:])
test_paths=sorted((category_root/'test').glob('*/*.png'))
assert len(train_paths)==196 and len(val_paths)==49 and len(test_paths)==124

RESAMPLE=getattr(Image,'Resampling',Image)
def load_image(path):
    with Image.open(path) as image:
        image=image.convert('RGB').resize((IMAGE_SIZE,IMAGE_SIZE),RESAMPLE.BILINEAR)
        array=np.asarray(image,dtype=np.float32).copy()/127.5-1.
    return torch.from_numpy(array).permute(2,0,1)
def load_mask(path):
    with Image.open(path) as image:
        array=np.asarray(image.convert('L').resize((IMAGE_SIZE,IMAGE_SIZE),RESAMPLE.NEAREST))>0
    return array

def mask_for_test(path):
    kind=path.parent.name
    return None if kind=='good' else category_root/'ground_truth'/kind/f'{path.stem}_mask.png'

for path in test_paths:
    mask=mask_for_test(path)
    if mask is not None:assert mask.is_file(),f'Missing mask {mask}'
print('Protocol:',len(train_paths),'train normal,',len(val_paths),'validation normal,',len(test_paths),'official test')


In [ ]:
assert CHECKPOINT.is_file(),'The 2,000-step checkpoint is missing from Google Drive.'
device=torch.device('cuda:0')
checkpoint=torch.load(CHECKPOINT,map_location=device,weights_only=False)
assert checkpoint['step']==2000,f"Expected step 2000, found {checkpoint['step']}"
assert checkpoint['author_commit']==author.COMMIT
model_config=checkpoint['model_config']
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
backbone=model_module.UDHVT(**model_config).to(device);model=Adapter(backbone)
model.load_state_dict(checkpoint['model']);model.eval()
diffusion=diffusion_ns['GaussianDiffusionModel'](
    [224,224],diffusion_ns['get_beta_schedule'](1000,'cosine'),img_channels=3,
    loss_type='l2',noise='4dsimplex',octave=6,frequency=64,persistence=.9,train=False)
# Compile Tsimplex before timing evaluation.
diffusion.noise_fn(torch.zeros(1,1,4,4,device=device),torch.tensor([5],device=device))
torch.cuda.reset_peak_memory_stats()
print('Loaded trained model with',sum(p.numel() for p in model.parameters()),'parameters.')


In [ ]:
def reconstruct_paths(paths,batch_size=BATCH_SIZE):
    reconstructions=[];inputs=[];seconds=[]
    for start in range(0,len(paths),batch_size):
        batch_paths=paths[start:start+batch_size]
        x=torch.stack([load_image(p) for p in batch_paths]).to(device)
        tick=time.perf_counter()
        with torch.inference_mode():
            recon=diffusion.forward_backward(model,x,None,see_whole_sequence=None,
                                              t_distance=T_DISTANCE,denoise_fn='noise_fn')
        torch.cuda.synchronize()
        seconds.append(time.perf_counter()-tick)
        inputs.append(x.cpu());reconstructions.append(recon.cpu())
        print(f'Reconstructed {min(start+len(batch_paths),len(paths))}/{len(paths)}',flush=True)
    return torch.cat(inputs),torch.cat(reconstructions),seconds

def residual_maps(inputs,recons):
    return (inputs-recons).square().mean(dim=1).numpy().astype(np.float32)


In [ ]:
# Threshold calibration uses only held-out normal images.
print('Reconstructing held-out normal validation images...')
val_inputs,val_recons,val_batch_seconds=reconstruct_paths(val_paths)
val_scores=residual_maps(val_inputs,val_recons)
threshold=float(np.quantile(val_scores.reshape(-1),1-TARGET_NORMAL_PIXEL_FPR,method='higher'))
observed_val_fpr=float(np.mean(val_scores>threshold))
calibration={
    'source':'49 held-out MVTec leather train/good images only',
    'target_normal_pixel_fpr':TARGET_NORMAL_PIXEL_FPR,
    'threshold':threshold,
    'observed_validation_pixel_fpr':observed_val_fpr,
    't_distance':T_DISTANCE,
}
(OUTPUT/'calibration.json').write_text(json.dumps(calibration,indent=2))
print(json.dumps(calibration,indent=2))


In [ ]:
# The threshold is now frozen. Test masks are loaded only for evaluation.
print('Running the locked official test evaluation...')
test_inputs,test_recons,test_batch_seconds=reconstruct_paths(test_paths)
test_scores=residual_maps(test_inputs,test_recons)
test_predictions=test_scores>threshold

ground_truth=[];image_labels=[];kinds=[]
for path in test_paths:
    kind=path.parent.name;kinds.append(kind);image_labels.append(kind!='good')
    mask_path=mask_for_test(path)
    ground_truth.append(np.zeros((IMAGE_SIZE,IMAGE_SIZE),bool) if mask_path is None else load_mask(mask_path))
ground_truth=np.stack(ground_truth)
image_labels=np.asarray(image_labels,bool)

from sklearn.metrics import roc_auc_score
pixel_auc=float(roc_auc_score(ground_truth.reshape(-1),test_scores.reshape(-1)))
# Robust high-percentile image score is declared in advance here and retained in the report.
image_scores=np.quantile(test_scores.reshape(len(test_paths),-1),.995,axis=1)
image_auc=float(roc_auc_score(image_labels,image_scores))

def confusion_metrics(pred,truth):
    tp=int(np.sum(pred&truth));fp=int(np.sum(pred&~truth));fn=int(np.sum(~pred&truth));tn=int(np.sum(~pred&~truth))
    return {'tp':tp,'fp':fp,'fn':fn,'tn':tn,
            'dice':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.,
            'iou':tp/(tp+fp+fn) if tp+fp+fn else 1.,
            'precision':tp/(tp+fp) if tp+fp else None,
            'recall':tp/(tp+fn) if tp+fn else None}

rows=[]
for i,path in enumerate(test_paths):
    metrics=confusion_metrics(test_predictions[i],ground_truth[i])
    rows.append({'image':path.relative_to(category_root).as_posix(),'kind':kinds[i],
                 'image_label':int(image_labels[i]),'image_score_p995':float(image_scores[i]),**metrics})
with (OUTPUT/'per_image.csv').open('w',newline='') as stream:
    writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)

defective=[row for row in rows if row['kind']!='good'];normal=[row for row in rows if row['kind']=='good']
normal_fpr=sum(row['fp'] for row in normal)/sum(row['fp']+row['tn'] for row in normal)
per_kind={}
for kind in sorted(set(kinds)):
    indices=np.asarray([value==kind for value in kinds])
    group=[rows[i] for i in np.flatnonzero(indices)]
    entry={'count':len(group)}
    if kind=='good':
        entry['pixel_false_positive_rate']=sum(x['fp'] for x in group)/sum(x['fp']+x['tn'] for x in group)
    else:
        entry.update({'mean_dice':float(np.mean([x['dice'] for x in group])),
                      'mean_iou':float(np.mean([x['iou'] for x in group])),
                      'pixel_auc':float(roc_auc_score(ground_truth[indices].reshape(-1),test_scores[indices].reshape(-1)))})
    per_kind[kind]=entry


In [ ]:
# Qualitative preview: one normal and one example per anomaly kind.
selected=[]
for kind in ['good','color','cut','fold','glue','poke']:
    selected.append(next(i for i,value in enumerate(kinds) if value==kind))
fig,axes=plt.subplots(len(selected),5,figsize=(14,3*len(selected)))
for row,index in enumerate(selected):
    original=((test_inputs[index].permute(1,2,0).numpy()+1)/2).clip(0,1)
    recon=((test_recons[index].permute(1,2,0).numpy()+1)/2).clip(0,1)
    panels=[original,recon,test_scores[index],ground_truth[index],test_predictions[index]]
    titles=[kinds[index],'Reconstruction','Squared residual','Ground truth','Prediction']
    for col,(panel,title) in enumerate(zip(panels,titles)):
        axes[row,col].imshow(panel,cmap=None if col<2 else 'magma' if col==2 else 'gray')
        axes[row,col].set_title(title);axes[row,col].axis('off')
fig.tight_layout();fig.savefig(OUTPUT/'evaluation_preview.png',dpi=150,bbox_inches='tight');plt.show()

report={
    'status':'passed','scope':'one_category_reduced_evaluation','paper_result_reproduced':False,
    'dataset':'MVTec AD','category':'leather','checkpoint_step':int(checkpoint['step']),
    'author_commit':author.COMMIT,'model_configuration':model_config,
    'train_normal_count':len(train_paths),'validation_normal_count':len(val_paths),
    'test_count':len(test_paths),'test_defective_count':int(image_labels.sum()),'test_normal_count':int((~image_labels).sum()),
    't_distance':T_DISTANCE,'t_distance_source':'Author fixed-time evaluation routine and paper partial-diffusion analysis; not selected on this test set.',
    'threshold':threshold,'threshold_source':'99.5th percentile of held-out normal validation residual pixels',
    'pixel_auroc':pixel_auc,'image_auroc_p995_score':image_auc,
    'mean_dice_defective_images':float(np.mean([x['dice'] for x in defective])),
    'mean_iou_defective_images':float(np.mean([x['iou'] for x in defective])),
    'mean_precision_defective_images':float(np.mean([x['precision'] or 0. for x in defective])),
    'mean_recall_defective_images':float(np.mean([x['recall'] or 0. for x in defective])),
    'normal_test_pixel_fpr':float(normal_fpr),'per_kind':per_kind,
    'median_validation_batch_seconds':float(np.median(val_batch_seconds)),
    'median_test_batch_seconds':float(np.median(test_batch_seconds)),
    'approx_test_seconds_per_image':float(sum(test_batch_seconds)/len(test_paths)),
    'peak_gpu_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
    'peak_gpu_reserved_gib':torch.cuda.max_memory_reserved()/2**30,
    'limitations':['One MVTec category','2,000 optimizer steps rather than paper 3,000 epochs','One reconstruction per image','Normal-only threshold protocol differs from test-optimized Dice selection','Architecture is the inspected SPE+DMHA+HFF+refinement configuration'],
}
(OUTPUT/'evaluation_report.json').write_text(json.dumps(report,indent=2))
# Compact half-precision maps are for qualitative audit; metrics above use float32 maps.
np.savez_compressed(OUTPUT/'test_maps_float16.npz',scores=test_scores.astype(np.float16),predictions=test_predictions,ground_truth=ground_truth)
archive=shutil.make_archive(str(WORK/'mvtec_leather_evaluation_results'),'zip',OUTPUT)
print(json.dumps(report,indent=2))
print('\nLOCKED TEST EVALUATION PASSED')
print('Download from Kaggle working files:',archive)
